<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Importing libraries.
</div>

In [ ]:
import h5py
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Handling the dataset.
</div>


In [ ]:
# ==========================================================
# Load RML2018
# ==========================================================
dataset_path = "datasets/GOLD_XYZ_OSC.0001_1024.hdf5"

MODULATIONS = [
    "OOK","ASK4","ASK8","BPSK","QPSK","PSK8","PSK16","PSK32",
    "APSK16","APSK32","APSK64","APSK128",
    "QAM16","QAM32","QAM64","QAM128","QAM256",
    "AM_SSB_WC","AM_SSB_SC","AM_DSB_WC","AM_DSB_SC",
    "FM","GMSK","OQPSK"
]

f = h5py.File(dataset_path, "r")

X_h5 = f["X"]
Y_h5 = f["Y"]
Z_h5 = f["Z"]

labels = np.argmax(Y_h5[:], axis=1)
snrs = Z_h5[:].flatten()

print("X:", X_h5.shape)
print("Y:", Y_h5.shape)
print("Z:", Z_h5.shape)
print("Classes:", len(MODULATIONS))

In [ ]:
# ==========================================================
# Train / Validation / Test Split
# ==========================================================
indices = np.arange(len(labels))

# Stratify jointly by modulation and SNR
stratify_key = np.array([
    f"{label}_{snr}" for label, snr in zip(labels, snrs)
])

idx_train, idx_temp = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=stratify_key
)

idx_val, idx_test = train_test_split(
    idx_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True,
    stratify=stratify_key[idx_temp]
)

print("Train:", len(idx_train))
print("Val:  ", len(idx_val))
print("Test: ", len(idx_test))

In [ ]:
# ==========================================================
# TensorFlow Dataset Generator
# ==========================================================
def make_rml2018_dataset(indices,
                         batch_size: int = 64,
                         shuffle: bool = False):

    indices = np.asarray(indices, dtype=np.int64)

    def generator():
        for idx in indices:
            idx = int(idx)

            x = X_h5[idx]                  # (1024, 2)
            y = np.argmax(Y_h5[idx])       # scalar label, read lazily

            x = x.T                        # (2, 1024)
            x = x[..., np.newaxis]         # (2, 1024, 1)

            yield x.astype(np.float32), np.int64(y)

    ds = tf.data.Dataset.from_generator(
        generator,
        output_signature=(
            tf.TensorSpec(shape=(2, 1024, 1), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int64),
        ),
    )

    if shuffle:
        ds = ds.shuffle(
            buffer_size=20000,
            reshuffle_each_iteration=True,
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds

In [ ]:
# ==========================================================
# Prepare TensorFlow Datasets
# ==========================================================
batch_size = 64

train_ds = make_rml2018_dataset(idx_train, batch_size=batch_size, shuffle=True)
val_ds   = make_rml2018_dataset(idx_val,   batch_size=batch_size, shuffle=False)
test_ds  = make_rml2018_dataset(idx_test,  batch_size=batch_size, shuffle=False)

In [ ]:
# ==========================================================
# Test One Batch
# ==========================================================
for xb, yb in train_ds.take(1):
    print("X batch:", xb.shape)
    print("y batch:", yb.shape)
    print("X dtype:", xb.dtype)
    print("y dtype:", yb.dtype)

<div style="background-color: #f9f9f9; color: #333333; border: 1px solid #cccccc; padding: 10px; border-radius: 5px;">
Training/validating/testing the Deep Learning model.
</div>


In [ ]:
# ==========================================================
# Build MCNet
# ==========================================================
from DeepLearning import build_mcnet

model = build_mcnet(
    input_shape=(2, 1024, 1),
    num_classes=24
)

model.summary()

In [ ]:
# ==========================================================
# Compile Network
# ==========================================================
optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.01,
    momentum=0.9
)

model.compile(
    optimizer=optimizer,
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# ==========================================================
# Callbacks
# ==========================================================
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="results/mcnet_rml2018_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.1,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    tf.keras.callbacks.CSVLogger(
        filename="results/mcnet_training_log.csv"
    )
]

In [ ]:
# ==========================================================
# Training Steps
# ==========================================================
steps_per_epoch = len(idx_train) // batch_size
validation_steps = len(idx_val) // batch_size

print("steps_per_epoch:", steps_per_epoch)
print("validation_steps:", validation_steps)

In [ ]:
# ==========================================================
# Training
# ==========================================================
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    steps_per_epoch=steps_per_epoch,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# ==========================================================
# Prediction with SNR
# ==========================================================
y_true = []
y_pred = []
snr_test = []

for idx in idx_test:
    idx = int(idx)

    x = X_h5[idx]                    # (1024, 2)
    y = np.argmax(Y_h5[idx])         # true class
    snr = int(Z_h5[idx][0])          # SNR

    x = x.T                          # (2, 1024)
    x = x[..., np.newaxis]           # (2, 1024, 1)
    x = np.expand_dims(x, axis=0)    # (1, 2, 1024, 1)

    probs = model.predict(x, verbose=0)
    pred = np.argmax(probs, axis=1)[0]

    y_true.append(y)
    y_pred.append(pred)
    snr_test.append(snr)

y_true = np.array(y_true)
y_pred = np.array(y_pred)
snr_test = np.array(snr_test)

In [ ]:
# ==========================================================
# Accuracy vs. SNR
# ==========================================================
snr_values = np.array(sorted(np.unique(snr_test)))

accuracy_snr = []

print("\nAccuracy vs. SNR")
print("-" * 35)

for snr in snr_values:

    mask = snr_test == snr

    acc = np.mean(y_pred[mask] == y_true[mask])
    accuracy_snr.append(acc)

    print(f"SNR = {snr:>4} dB | Accuracy = {acc:.4f}")

In [ ]:
# ==========================================================
# Confusion Matrix and Report
# ==========================================================
from sklearn.metrics import classification_report, confusion_matrix

class_names = MODULATIONS

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

cm = confusion_matrix(y_true, y_pred)

print(cm)